# Task 2
Requesting data from the Baltic Transparency-Dashboard API, ploting graphs from the retrieved data and assessing regulation activities.

In [11]:
import io
import json
import zipfile

import pandas as pd
import requests

### Task 2.1.
Requesting data from the Baltic Transparency-Dashboard API.

In [12]:
api_url = "https://api-baltic.transparency-dashboard.eu/api/v1/export-multiple"
request_parameters = {
    "id": "activations_afrr,imbalance_volumes_v2",
    "start_date": "2025-09-22T00:00",
    "end_date": "2025-09-22T23:59",
    "output_time_zone": "EET",
    "output_format": "json",
    "json_header_groups": 0,
}
target_local_date = pd.Timestamp("2025-09-22").date()
local_timezone = "Europe/Tallinn"

response = requests.get(
    api_url,
    params=request_parameters,
    headers={"accept": "application/zip"},
    timeout=30,
)
response.raise_for_status()

# The API returns one JSON document per requested dataset inside a ZIP archive.
exports = {}
with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
    for filename in archive.namelist():
        dataset = json.loads(archive.read(filename))
        exports[dataset["id"]] = dataset


def export_to_dataframe(dataset):
    column_names = []
    for column in dataset["columns"]:
        group = column.get("group_level_0")
        label = column.get("label")
        column_names.append(": ".join(str(part) for part in [group, label] if part))

    rows = []
    for interval in dataset["timeseries"]:
        # Convert UTC timestamps to local time, then display them without offsets.
        interval_start = pd.Timestamp(interval["from"]).tz_convert(local_timezone).tz_localize(None)
        interval_end = pd.Timestamp(interval["to"]).tz_convert(local_timezone).tz_localize(None)
        row = {"from": interval_start, "to": interval_end}
        row.update(dict(zip(column_names, interval["values"])))
        rows.append(row)

    dataframe = pd.DataFrame(rows).set_index("from").sort_index()
    return dataframe[dataframe.index.date == target_local_date]


activations_df = export_to_dataframe(exports["activations_afrr"])
imbalance_df = export_to_dataframe(exports["imbalance_volumes_v2"])

print(f"Retrieved {len(activations_df)} local intervals for {target_local_date}")
print(f"Local interval range: {activations_df.index.min()} to {activations_df['to'].max()}")
display(activations_df.head())
display(imbalance_df.head())

Retrieved 96 local intervals for 2025-09-22
Local interval range: 2025-09-22 00:00:00 to 2025-09-23 00:00:00


,to,Estonia: Upward,Estonia: Downward,Latvia: Upward,Latvia: Downward,Lithuania: Upward,Lithuania: Downward
from,,,,,,,
2025-09-22 00:00:00,2025-09-22 00:15:00,0.002,1.570,0.009,0.891,0.00,6.30
2025-09-22 00:15:00,2025-09-22 00:30:00,0.027,0.641,0.540,0.089,0.25,0.72
2025-09-22 00:30:00,2025-09-22 00:45:00,0.130,0.760,1.257,0.146,0.10,0.78
2025-09-22 00:45:00,2025-09-22 01:00:00,0.022,0.689,0.017,0.955,0.00,12.98
2025-09-22 01:00:00,2025-09-22 01:15:00,1.014,0.344,3.260,0.377,0.00,1.55


,to,Estonia,Latvia,Lithuania
from,,,,
2025-09-22 00:00:00,2025-09-22 00:15:00,10.711,-2.746,5.011
2025-09-22 00:15:00,2025-09-22 00:30:00,13.394,-0.702,-0.038
2025-09-22 00:30:00,2025-09-22 00:45:00,15.316,1.999,-9.780
2025-09-22 00:45:00,2025-09-22 01:00:00,14.849,6.053,2.289
2025-09-22 01:00:00,2025-09-22 01:15:00,15.610,-0.608,3.688


In [14]:
imbalance_df.head(96)

,to,Estonia,Latvia,Lithuania
from,,,,
2025-09-22 00:00:00,2025-09-22 00:15:00,10.711,-2.746,5.011
2025-09-22 00:15:00,2025-09-22 00:30:00,13.394,-0.702,-0.038
2025-09-22 00:30:00,2025-09-22 00:45:00,15.316,1.999,-9.780
2025-09-22 00:45:00,2025-09-22 01:00:00,14.849,6.053,2.289
2025-09-22 01:00:00,2025-09-22 01:15:00,15.610,-0.608,3.688
...,...,...,...,...
2025-09-22 22:45:00,2025-09-22 23:00:00,16.679,6.470,24.055
2025-09-22 23:00:00,2025-09-22 23:15:00,10.326,-5.166,-3.967
2025-09-22 23:15:00,2025-09-22 23:30:00,15.779,0.347,-6.481
